
### Wrapper for `pszp.py`

This notebook contains a simple programme that takes the forced photometry table of a Pan-STARRS observation (downloaded from the Pan-STARRS eyeballing pages), and does the photometry correction to account for the zero-point correction needed to finalise photometry of targeted observations.

**Actually it does the first part; it determines what the zero-point corrections should be. The second part (`2-applying_zero-points.ipynb`) pairs these with the forced photometry table, and outputs final photometry.**

To do this, the .cmf files of the observations are needed $-$ instructions on how to download these are here: https://psweb.mp.qub.ac.uk/psat-lv-wiki/index.php/Zeropoint_Corrections_for_Pan-STARRS_difference_images


The `pszp.py` script lets one determine the zero-point corrections for the frames based on the information in the .cmf files. The script is very particular about how the different exposures are grouped (e.g., can't have files grouped in a shared directory with multiple exposures in the same filter). The previous script (`0-CMF_rearranger.ipynb`) **should** have correctly reorganised the CMF files to play nicely with this wrapper; if not, check the directory structure!


In [ ]:

""" loading relevant packages """

from glob import glob


In [ ]:

""" Defining filepath information """

# specify the filepath to the .cmf files that we want to perform the zero-point correction to
path_to_CMFs = "../Bundles/CMFs"


In [ ]:

""" Removing previous versions of `ref_stars.dat` and `offsets.txt` """

cmd = f"rm {path_to_CMFs}/offsets.txt"
print(f"\n\nThis is the command I am about to run...\n{cmd}")

# running the command
! {cmd}


cmd = f"rm {path_to_CMFs}/ref_stars.dat"
print(f"\n\nThis is the command I am about to run...\n{cmd}")

# running the command
! {cmd}


In [ ]:

""" grabbing list of CMFs """

filepaths = sorted(glob(f"{path_to_CMFs}/*"))

filepaths
# len(filepaths)


In [ ]:

def read_metadata():
    """ useful for reading in transient metadata """
    
    data = {}

    with open("metadata.txt", "r") as f:
        for line in f:
            line = line.strip()

            # Skip empty lines and headers
            if not line or line.startswith("#"):
                continue

            key, value = line.split(":", 1)

            data[key.strip()] = value.strip()

    transient_name = data["transient_name"]
    ra = data["RA"]
    dec = data["Dec"]

    print("transient_name, ra, dec")
    print(transient_name, ra, dec)
    
    return transient_name, ra, dec


In [ ]:

""" iterating through all CMF files to compute a zero-point offset """

# NOTE. I know the current directory structure is inefficient computationally, but
# it is the implementation that is most 'foolproof', and the one that requires the
# least effort; i.e., I don't have to concern myself with what filters each exposure
# utilised. There cannot be duplicate filters for the same epoch, which requires some
# effort to avoid; the current implementation is completely filter-agnostic (albeit
# somewhat inefficient).

transient_name, ra, dec = read_metadata()

for filepath in filepaths:

    # this clips the filepath to remove "Bundles" from filepath (needed to run pszp.py)
    subfilepath = filepath.split("Bundles/")[1]
    print(subfilepath)

    # define the python command with relevant executables
    cmd = f"python ../pszp.py -d {subfilepath} -c {ra} {dec}"
    print(f"\n\nThis is the command I am about to run...\n{cmd}\n\n")

    # running the command
    ! {cmd}
